In [ ]:
!pip install mediapipe==0.10.14 opencv-python-headless --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 10.4 MB/s eta 0:00:00


In [ ]:
!pip install -q kaggle

import os
import json

KAGGLE_USERNAME="M*********1"
KAGGLE_KEY="KGAT_***************************e"

kaggle_dict = {
    "username": KAGGLE_USERNAME,
    "key": KAGGLE_KEY
}

with open("kaggle.json", "w") as f:
  json.dump(kaggle_dict, f)


!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("kaggle configured ✅")

kaggle configured ✅


In [ ]:
!kaggle datasets download -d niten19/face-shape-dataset


Dataset URL: https://www.kaggle.com/datasets/niten19/face-shape-dataset
License(s): CC0-1.0
face-shape-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
!unzip face-shape-dataset.zip

In [ ]:
import cv2
import numpy as np
import os
import random
from collections import Counter
import mediapipe as mp

print("Mediapipe version:", mp.__version__)

mp_face_mesh = mp.solutions.face_mesh

face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=True,
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5
)

In [ ]:


DATA_PATH = "/content/FaceShape Dataset/training_set"

print("Exists:", os.path.exists(DATA_PATH))
print("Folders:", os.listdir(DATA_PATH))

In [ ]:
def get_landmarks(img):
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(img_rgb)

    if not results.multi_face_landmarks:
        return None

    coords = []
    for lm in results.multi_face_landmarks[0].landmark:
        coords.extend([lm.x, lm.y, lm.z])

    return np.array(coords)

In [ ]:
def process_face_shape(path, limit=5000):
    X, y = [], []

    labels = [
        l for l in os.listdir(path)
        if os.path.isdir(os.path.join(path, l))
    ]

    per_class = limit // len(labels)

    print("🚀 Processing Face Shape Dataset...")

    for label in labels:
        label_path = os.path.join(path, label)

        files = os.listdir(label_path)
        random.shuffle(files)

        count = 0

        print(f"\n📁 Label: {label}")
        print("Files found:", len(files))

        for img_name in files:
            if not img_name.lower().endswith((".jpg", ".png", ".jpeg")):
                continue

            img_path = os.path.join(label_path, img_name)

            print("➡️ Trying:", img_path)

            img = cv2.imread(img_path)

            if img is None:
                print("❌ Image read failed")
                continue

            # 🔥 IMPORTANT CHANGE
            img = cv2.resize(img, (512, 512))

            landmarks = get_landmarks(img)

            if landmarks is None:
                print("❌ Face not detected")
                continue

            print("✅ Face detected")

            X.append(landmarks)
            y.append(label)

            count += 1

            if count >= per_class:
                break

        print(f"✅ Collected for {label}: {count}")

    X = np.array(X)
    y = np.array(y)

    from collections import Counter
    print("\n📊 Distribution:", Counter(y))
    print("📐 Shape:", X.shape)

    np.save("face_landmarks.npy", X)
    np.save("face_labels.npy", y)

    print("\n✅ DONE")

In [ ]:

test_img = "/content/FaceShape Dataset/training_set/Round/round (1).jpg"

img = cv2.imread(test_img)
img = cv2.resize(img, (512, 512))

lm = get_landmarks(img)

print("Landmarks detected:", lm is not None)

In [ ]:
DATA_PATH = "/content/FaceShape Dataset/training_set"

process_face_shape(DATA_PATH, limit=5000)

In [ ]:
from google.colab import files

files.download("face_landmarks.npy")
files.download("face_labels.npy")